# Лабораторная работа 4
### Выполнил: Стрельников Никита, группа 6401-010302D
### Вариант 3: набор данных CIFAR-10

Tensorflow 2.x

1) Подготовка данных

2) Использование Keras Model API

3) Использование Keras Sequential + Functional API

https://www.tensorflow.org/tutorials

Для выполнения лабораторной работы необходимо установить tensorflow версии 2.0 или выше .

Рекомендуется использовать возможности Colab'а по обучению моделей на GPU.



In [1]:
import os
import tensorflow as tf
import numpy as np
import math
import timeit
import matplotlib.pyplot as plt

# Используем для обучения GPU, если доступен
if tf.config.list_physical_devices('GPU'):
    device = '/GPU:0'
else:
    device = '/CPU:0'
print("Определен ", device)

%matplotlib inline

Определен  /GPU:0


# Подготовка данных
Загрузите набор данных из предыдущей лабораторной работы.

In [2]:
def load_cifar10(num_training=49000, num_validation=1000, num_test=10000):
    """
    Fetch the CIFAR-10 dataset from the web and perform preprocessing to prepare
    it for the two-layer neural net classifier. These are the same steps as
    we used for the SVM, but condensed to a single function.
    """
    # Load the raw CIFAR-10 dataset and use appropriate data types and shapes
    cifar10 = tf.keras.datasets.cifar10.load_data()
    (X_train, y_train), (X_test, y_test) = cifar10
    X_train = np.asarray(X_train, dtype=np.float32)
    y_train = np.asarray(y_train, dtype=np.int32).flatten()
    X_test = np.asarray(X_test, dtype=np.float32)
    y_test = np.asarray(y_test, dtype=np.int32).flatten()

    # Subsample the data
    mask = range(num_training, num_training + num_validation)
    X_val = X_train[mask]
    y_val = y_train[mask]
    mask = range(num_training)
    X_train = X_train[mask]
    y_train = y_train[mask]
    mask = range(num_test)
    X_test = X_test[mask]
    y_test = y_test[mask]

    # Normalize the data: subtract the mean pixel and divide by std
    mean_pixel = X_train.mean(axis=(0, 1, 2), keepdims=True)
    std_pixel = X_train.std(axis=(0, 1, 2), keepdims=True)
    X_train = (X_train - mean_pixel) / std_pixel
    X_val = (X_val - mean_pixel) / std_pixel
    X_test = (X_test - mean_pixel) / std_pixel

    return X_train, y_train, X_val, y_val, X_test, y_test

# If there are errors with SSL downloading involving self-signed certificates,
# it may be that your Python version was recently installed on the current machine.
# See: https://github.com/tensorflow/tensorflow/issues/10779
# To fix, run the command: /Applications/Python\ 3.7/Install\ Certificates.command
#   ...replacing paths as necessary.

# Invoke the above function to get our data.
NHW = (0, 1, 2)
X_train, y_train, X_val, y_val, X_test, y_test = load_cifar10()
print('Train data shape: ', X_train.shape)
print('Train labels shape: ', y_train.shape, y_train.dtype)
print('Validation data shape: ', X_val.shape)
print('Validation labels shape: ', y_val.shape)
print('Test data shape: ', X_test.shape)
print('Test labels shape: ', y_test.shape)

170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 4s 0us/step
Train data shape:  (49000, 32, 32, 3)
Train labels shape:  (49000,) int32
Validation data shape:  (1000, 32, 32, 3)
Validation labels shape:  (1000,)
Test data shape:  (10000, 32, 32, 3)
Test labels shape:  (10000,)


In [3]:
class Dataset(object):
    def __init__(self, X, y, batch_size, shuffle=False):
        """
        Construct a Dataset object to iterate over data X and labels y

        Inputs:
        - X: Numpy array of data, of any shape
        - y: Numpy array of labels, of any shape but with y.shape[0] == X.shape[0]
        - batch_size: Integer giving number of elements per minibatch
        - shuffle: (optional) Boolean, whether to shuffle the data on each epoch
        """
        assert X.shape[0] == y.shape[0], 'Got different numbers of data and labels'
        self.X, self.y = X, y
        self.batch_size, self.shuffle = batch_size, shuffle

    def __iter__(self):
        N, B = self.X.shape[0], self.batch_size
        idxs = np.arange(N)
        if self.shuffle:
            np.random.shuffle(idxs)
        return iter((self.X[i:i+B], self.y[i:i+B]) for i in range(0, N, B))


train_dset = Dataset(X_train, y_train, batch_size=64, shuffle=True)
val_dset = Dataset(X_val, y_val, batch_size=64, shuffle=False)
test_dset = Dataset(X_test, y_test, batch_size=64)

In [4]:
# We can iterate through a dataset like this:
for t, (x, y) in enumerate(train_dset):
    print(t, x.shape, y.shape)
    if t > 5: break

0 (64, 32, 32, 3) (64,)
1 (64, 32, 32, 3) (64,)
2 (64, 32, 32, 3) (64,)
3 (64, 32, 32, 3) (64,)
4 (64, 32, 32, 3) (64,)
5 (64, 32, 32, 3) (64,)
6 (64, 32, 32, 3) (64,)


#  Keras Model Subclassing API


Для реализации собственной модели с помощью Keras Model Subclassing API необходимо выполнить следующие шаги:

1) Определить новый класс, который является наследником tf.keras.Model.

2) В методе __init__() определить все необходимые слои из модуля tf.keras.layer

3) Реализовать прямой проход в методе call() на основе слоев, объявленных в __init__()

Ниже приведен пример использования keras API для определения двухслойной полносвязной сети.

https://www.tensorflow.org/versions/r2.0/api_docs/python/tf/keras

In [5]:
class TwoLayerFC(tf.keras.Model):
    def __init__(self, hidden_size, num_classes):
        super(TwoLayerFC, self).__init__()
        initializer = tf.initializers.VarianceScaling(scale=2.0)
        self.fc1 = tf.keras.layers.Dense(hidden_size, activation='relu',
                                   kernel_initializer=initializer)
        self.fc2 = tf.keras.layers.Dense(num_classes, activation='softmax',
                                   kernel_initializer=initializer)
        self.flatten = tf.keras.layers.Flatten()

    def call(self, x, training=False):
        x = self.flatten(x)
        x = self.fc1(x)
        x = self.fc2(x)
        return x


def test_TwoLayerFC():
    """ A small unit test to exercise the TwoLayerFC model above. """
    input_size, hidden_size, num_classes = 50, 42, 10
    x = tf.zeros((64, input_size))
    model = TwoLayerFC(hidden_size, num_classes)
    with tf.device(device):
        scores = model(x)
        print(scores.shape)

test_TwoLayerFC()

(64, 10)


Реализуйте трехслойную CNN для вашей задачи классификации.

Архитектура сети:
    
1. Сверточный слой (5 x 5 kernels, zero-padding = 'same')
2. Функция активации ReLU
3. Сверточный слой (3 x 3 kernels, zero-padding = 'same')
4. Функция активации ReLU
5. Полносвязный слой
6. Функция активации Softmax

https://www.tensorflow.org/versions/r2.0/api_docs/python/tf/keras/layers/Conv2D

https://www.tensorflow.org/versions/r2.0/api_docs/python/tf/keras/layers/Dense

In [6]:
class ThreeLayerConvNet(tf.keras.Model):
    def __init__(self, channel_1, channel_2, num_classes):
        super(ThreeLayerConvNet, self).__init__()
        ########################################################################
        # TODO: Implement the __init__ method for a three-layer ConvNet. You   #
        # should instantiate layer objects to be used in the forward pass.     #
        ########################################################################
        # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

        initializer = tf.initializers.VarianceScaling(scale=2.0)

        self.conv1 = tf.keras.layers.Conv2D(
            filters=channel_1,
            kernel_size=5,
            padding='same',
            activation='relu',
            kernel_initializer=initializer
        )

        self.conv2 = tf.keras.layers.Conv2D(
            filters=channel_2,
            kernel_size=3,
            padding='same',
            activation='relu',
            kernel_initializer=initializer
        )

        self.flatten = tf.keras.layers.Flatten()

        self.fc = tf.keras.layers.Dense(
            num_classes,
            activation='softmax',
            kernel_initializer=initializer
        )

        # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
        ########################################################################
        #                           END OF YOUR CODE                           #
        ########################################################################

    def call(self, x, training=False):
        scores = None
        ########################################################################
        # TODO: Implement the forward pass for a three-layer ConvNet. You      #
        # should use the layer objects defined in the __init__ method.         #
        ########################################################################
        # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

        x = self.conv1(x)
        x = self.conv2(x)
        x = self.flatten(x)
        scores = self.fc(x)

        # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
        ########################################################################
        #                           END OF YOUR CODE                           #
        ########################################################################
        return scores

In [7]:
def test_ThreeLayerConvNet():
    channel_1, channel_2, num_classes = 12, 8, 10
    model = ThreeLayerConvNet(channel_1, channel_2, num_classes)
    with tf.device(device):
        x = tf.zeros((64, 3, 32, 32))
        scores = model(x)
        print(scores.shape)

test_ThreeLayerConvNet()

(64, 10)


Пример реализации процесса обучения:

In [8]:
def train_part34(model_init_fn, optimizer_init_fn, num_epochs=1, is_training=False):
    """
    Simple training loop for use with models defined using tf.keras. It trains
    a model for one epoch on the CIFAR-10 training set and periodically checks
    accuracy on the CIFAR-10 validation set.

    Inputs:
    - model_init_fn: A function that takes no parameters; when called it
      constructs the model we want to train: model = model_init_fn()
    - optimizer_init_fn: A function which takes no parameters; when called it
      constructs the Optimizer object we will use to optimize the model:
      optimizer = optimizer_init_fn()
    - num_epochs: The number of epochs to train for

    Returns: Nothing, but prints progress during trainingn
    """
    with tf.device(device):


        loss_fn = tf.keras.losses.SparseCategoricalCrossentropy()

        model = model_init_fn()
        optimizer = optimizer_init_fn()

        train_loss = tf.keras.metrics.Mean(name='train_loss')
        train_accuracy = tf.keras.metrics.SparseCategoricalAccuracy(name='train_accuracy')

        val_loss = tf.keras.metrics.Mean(name='val_loss')
        val_accuracy = tf.keras.metrics.SparseCategoricalAccuracy(name='val_accuracy')

        t = 0
        for epoch in range(num_epochs):

            # Reset the metrics - https://www.tensorflow.org/alpha/guide/migration_guide#new-style_metrics
            train_loss.reset_state()
            train_accuracy.reset_state()

            for x_np, y_np in train_dset:
                with tf.GradientTape() as tape:

                    # Use the model function to build the forward pass.
                    scores = model(x_np, training=is_training)
                    loss = loss_fn(y_np, scores)

                    gradients = tape.gradient(loss, model.trainable_variables)
                    optimizer.apply_gradients(zip(gradients, model.trainable_variables))

                    # Update the metrics
                    train_loss.update_state(loss)
                    train_accuracy.update_state(y_np, scores)

                    if t % print_every == 0:
                        val_loss.reset_state()
                        val_accuracy.reset_state()
                        for test_x, test_y in val_dset:
                            # During validation at end of epoch, training set to False
                            prediction = model(test_x, training=False)
                            t_loss = loss_fn(test_y, prediction)

                            val_loss.update_state(t_loss)
                            val_accuracy.update_state(test_y, prediction)

                        template = 'Iteration {}, Epoch {}, Loss: {}, Accuracy: {}, Val Loss: {}, Val Accuracy: {}'
                        print (template.format(t, epoch+1,
                                             train_loss.result(),
                                             train_accuracy.result()*100,
                                             val_loss.result(),
                                             val_accuracy.result()*100))
                    t += 1

In [9]:
hidden_size, num_classes = 4000, 10
learning_rate = 1e-2
print_every = 100

def model_init_fn():
    return TwoLayerFC(hidden_size, num_classes)

def optimizer_init_fn():
    return tf.keras.optimizers.SGD(learning_rate=learning_rate)

train_part34(model_init_fn, optimizer_init_fn)

Iteration 0, Epoch 1, Loss: 3.038705348968506, Accuracy: 10.9375, Val Loss: 3.111675977706909, Val Accuracy: 12.5
Iteration 100, Epoch 1, Loss: 2.224102258682251, Accuracy: 29.099628448486328, Val Loss: 1.857177972793579, Val Accuracy: 38.5
Iteration 200, Epoch 1, Loss: 2.070955991744995, Accuracy: 32.6414794921875, Val Loss: 1.7793470621109009, Val Accuracy: 40.599998474121094
Iteration 300, Epoch 1, Loss: 1.9930217266082764, Accuracy: 34.44248580932617, Val Loss: 1.8504387140274048, Val Accuracy: 38.20000076293945
Iteration 400, Epoch 1, Loss: 1.925253987312317, Accuracy: 36.155704498291016, Val Loss: 1.6970088481903076, Val Accuracy: 42.29999923706055
Iteration 500, Epoch 1, Loss: 1.881978154182434, Accuracy: 37.278568267822266, Val Loss: 1.6247178316116333, Val Accuracy: 43.0
Iteration 600, Epoch 1, Loss: 1.8503187894821167, Accuracy: 38.191558837890625, Val Loss: 1.697993516921997, Val Accuracy: 40.70000076293945
Iteration 700, Epoch 1, Loss: 1.8250210285186768, Accuracy: 38.82400

Обучите трехслойную CNN. В tf.keras.optimizers.SGD укажите Nesterov momentum = 0.9 .

https://www.tensorflow.org/versions/r2.0/api_docs/python/tf/optimizers/SGD

Значение accuracy на валидационной выборке после 1 эпохи обучения должно быть > 50% .

In [10]:
learning_rate = 3e-3
channel_1, channel_2, num_classes = 32, 16, 10

def model_init_fn():
    model = None
    ############################################################################
    # TODO: Complete the implementation of model_fn.                           #
    ############################################################################
    # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

    model = ThreeLayerConvNet(channel_1, channel_2, num_classes)

    # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
    ############################################################################
    #                           END OF YOUR CODE                               #
    ############################################################################
    return model

def optimizer_init_fn():
    optimizer = None
    ############################################################################
    # TODO: Complete the implementation of model_fn.                           #
    ############################################################################
    # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

    optimizer = tf.keras.optimizers.SGD(
        learning_rate=learning_rate,
        momentum=0.9,
        nesterov=True
    )

    # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
    ############################################################################
    #                           END OF YOUR CODE                               #
    ############################################################################
    return optimizer

train_part34(model_init_fn, optimizer_init_fn)

Iteration 0, Epoch 1, Loss: 3.1812567710876465, Accuracy: 15.625, Val Loss: 8.265109062194824, Val Accuracy: 12.700000762939453
Iteration 100, Epoch 1, Loss: 2.1381893157958984, Accuracy: 29.2852725982666, Val Loss: 1.7580170631408691, Val Accuracy: 39.20000076293945
Iteration 200, Epoch 1, Loss: 1.9050679206848145, Accuracy: 35.4710807800293, Val Loss: 1.6115264892578125, Val Accuracy: 43.5
Iteration 300, Epoch 1, Loss: 1.7953470945358276, Accuracy: 38.283843994140625, Val Loss: 1.5210589170455933, Val Accuracy: 47.60000228881836
Iteration 400, Epoch 1, Loss: 1.7074347734451294, Accuracy: 40.73020553588867, Val Loss: 1.4629608392715454, Val Accuracy: 47.400001525878906
Iteration 500, Epoch 1, Loss: 1.6476545333862305, Accuracy: 42.46818542480469, Val Loss: 1.3987808227539062, Val Accuracy: 50.30000305175781
Iteration 600, Epoch 1, Loss: 1.6080328226089478, Accuracy: 43.7239990234375, Val Loss: 1.3608344793319702, Val Accuracy: 52.10000228881836
Iteration 700, Epoch 1, Loss: 1.57141613

# Использование Keras Sequential API для реализации последовательных моделей.

Пример для полносвязной сети:

In [11]:
learning_rate = 1e-2

def model_init_fn():
    input_shape = (32, 32, 3)
    hidden_layer_size, num_classes = 4000, 10
    initializer = tf.initializers.VarianceScaling(scale=2.0)
    layers = [
        tf.keras.layers.Flatten(input_shape=input_shape),
        tf.keras.layers.Dense(hidden_layer_size, activation='relu',
                              kernel_initializer=initializer),
        tf.keras.layers.Dense(num_classes, activation='softmax',
                              kernel_initializer=initializer),
    ]
    model = tf.keras.Sequential(layers)
    return model

def optimizer_init_fn():
    return tf.keras.optimizers.SGD(learning_rate=learning_rate)

train_part34(model_init_fn, optimizer_init_fn)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Iteration 0, Epoch 1, Loss: 2.9271349906921387, Accuracy: 9.375, Val Loss: 2.689979076385498, Val Accuracy: 14.90000057220459
Iteration 100, Epoch 1, Loss: 2.2186119556427, Accuracy: 28.573638916015625, Val Loss: 1.8868167400360107, Val Accuracy: 37.5
Iteration 200, Epoch 1, Loss: 2.059389352798462, Accuracy: 32.40049743652344, Val Loss: 1.845069169998169, Val Accuracy: 40.0
Iteration 300, Epoch 1, Loss: 1.987233281135559, Accuracy: 34.22446060180664, Val Loss: 1.883324384689331, Val Accuracy: 35.70000076293945
Iteration 400, Epoch 1, Loss: 1.9196343421936035, Accuracy: 36.00763702392578, Val Loss: 1.730776071548462, Val Accuracy: 42.29999923706055
Iteration 500, Epoch 1, Loss: 1.876853346824646, Accuracy: 37.17252731323242, Val Loss: 1.6603844165802002, Val Accuracy: 43.20000076293945
Iteration 600, Epoch 1, Loss: 1.848666787147522, Accuracy: 37.85357666015625, Val Loss: 1.6709617376327515, Val Accuracy: 42.5
Iteration 700, Epoch 1, Loss: 1.8221352100372314, Accuracy: 38.5565261840820

Альтернативный менее гибкий способ обучения:

In [12]:
model = model_init_fn()
model.compile(optimizer=tf.keras.optimizers.SGD(learning_rate=learning_rate),
              loss='sparse_categorical_crossentropy',
              metrics=[tf.keras.metrics.sparse_categorical_accuracy])
model.fit(X_train, y_train, batch_size=64, epochs=1, validation_data=(X_val, y_val))
model.evaluate(X_test, y_test)

766/766 ━━━━━━━━━━━━━━━━━━━━ 6s 6ms/step - loss: 1.8281 - sparse_categorical_accuracy: 0.3867 - val_loss: 1.7249 - val_sparse_categorical_accuracy: 0.4270
313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - loss: 1.7029 - sparse_categorical_accuracy: 0.4178


[1.7029004096984863, 0.41780000925064087]

Перепишите реализацию трехслойной CNN с помощью tf.keras.Sequential API . Обучите модель двумя способами.

In [13]:
def model_init_fn():
    model = None
    ############################################################################
    # TODO: Construct a three-layer ConvNet using tf.keras.Sequential.         #
    ############################################################################
    # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

    initializer = tf.initializers.VarianceScaling(scale=2.0)

    model = tf.keras.Sequential([
        tf.keras.layers.Conv2D(
            filters=channel_1,
            kernel_size=5,
            padding='same',
            activation='relu',
            kernel_initializer=initializer,
            input_shape=(32, 32, 3)
        ),
        tf.keras.layers.Conv2D(
            filters=channel_2,
            kernel_size=3,
            padding='same',
            activation='relu',
            kernel_initializer=initializer
        ),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(
            num_classes,
            activation='softmax',
            kernel_initializer=initializer
        )
    ])

    # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
    ############################################################################
    #                            END OF YOUR CODE                              #
    ############################################################################
    return model

learning_rate = 5e-4
def optimizer_init_fn():
    optimizer = None
    ############################################################################
    # TODO: Complete the implementation of model_fn.                           #
    ############################################################################
    # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

    optimizer = tf.keras.optimizers.SGD(
        learning_rate=learning_rate,
        momentum=0.9,
        nesterov=True
    )

    # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
    ############################################################################
    #                           END OF YOUR CODE                               #
    ############################################################################
    return optimizer

train_part34(model_init_fn, optimizer_init_fn)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Iteration 0, Epoch 1, Loss: 2.79197359085083, Accuracy: 7.8125, Val Loss: 2.6640706062316895, Val Accuracy: 10.40000057220459
Iteration 100, Epoch 1, Loss: 2.025360345840454, Accuracy: 28.63551902770996, Val Loss: 1.8007934093475342, Val Accuracy: 37.0
Iteration 200, Epoch 1, Loss: 1.8889027833938599, Accuracy: 33.550994873046875, Val Loss: 1.662527084350586, Val Accuracy: 42.69999694824219
Iteration 300, Epoch 1, Loss: 1.8146110773086548, Accuracy: 36.139949798583984, Val Loss: 1.6158993244171143, Val Accuracy: 45.0
Iteration 400, Epoch 1, Loss: 1.7526508569717407, Accuracy: 38.2325439453125, Val Loss: 1.569157600402832, Val Accuracy: 46.39999771118164
Iteration 500, Epoch 1, Loss: 1.7109241485595703, Accuracy: 39.61764144897461, Val Loss: 1.533124327659607, Val Accuracy: 45.599998474121094
Iteration 600, Epoch 1, Loss: 1.6852277517318726, Accuracy: 40.54440689086914, Val Loss: 1.5086880922317505, Val Accuracy: 48.79999923706055
Iteration 700, Epoch 1, Loss: 1.659603238105774, Accurac

In [14]:
model = model_init_fn()
model.compile(optimizer='sgd',
              loss='sparse_categorical_crossentropy',
              metrics=[tf.keras.metrics.sparse_categorical_accuracy])
model.fit(X_train, y_train, batch_size=64, epochs=1, validation_data=(X_val, y_val))
model.evaluate(X_test, y_test)

766/766 ━━━━━━━━━━━━━━━━━━━━ 6s 6ms/step - loss: 1.6136 - sparse_categorical_accuracy: 0.4399 - val_loss: 1.3972 - val_sparse_categorical_accuracy: 0.5000
313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - loss: 1.3982 - sparse_categorical_accuracy: 0.4965


[1.3982136249542236, 0.4964999854564667]

# Использование Keras Functional API

Для реализации более сложных архитектур сети с несколькими входами/выходами, повторным использованием слоев, "остаточными" связями (residual connections) необходимо явно указать входные и выходные тензоры.

Ниже представлен пример для полносвязной сети.

In [15]:
def two_layer_fc_functional(input_shape, hidden_size, num_classes):
    initializer = tf.initializers.VarianceScaling(scale=2.0)
    inputs = tf.keras.Input(shape=input_shape)
    flattened_inputs = tf.keras.layers.Flatten()(inputs)
    fc1_output = tf.keras.layers.Dense(hidden_size, activation='relu',
                                 kernel_initializer=initializer)(flattened_inputs)
    scores = tf.keras.layers.Dense(num_classes, activation='softmax',
                             kernel_initializer=initializer)(fc1_output)

    # Instantiate the model given inputs and outputs.
    model = tf.keras.Model(inputs=inputs, outputs=scores)
    return model

def test_two_layer_fc_functional():
    """ A small unit test to exercise the TwoLayerFC model above. """
    input_size, hidden_size, num_classes = 50, 42, 10
    input_shape = (50,)

    x = tf.zeros((64, input_size))
    model = two_layer_fc_functional(input_shape, hidden_size, num_classes)

    with tf.device(device):
        scores = model(x)
        print(scores.shape)

test_two_layer_fc_functional()

(64, 10)


In [16]:
input_shape = (32, 32, 3)
hidden_size, num_classes = 4000, 10
learning_rate = 1e-2

def model_init_fn():
    return two_layer_fc_functional(input_shape, hidden_size, num_classes)

def optimizer_init_fn():
    return tf.keras.optimizers.SGD(learning_rate=learning_rate)

train_part34(model_init_fn, optimizer_init_fn)

Iteration 0, Epoch 1, Loss: 3.139242172241211, Accuracy: 12.5, Val Loss: 2.912170648574829, Val Accuracy: 12.800000190734863
Iteration 100, Epoch 1, Loss: 2.2272346019744873, Accuracy: 28.83663558959961, Val Loss: 1.8471887111663818, Val Accuracy: 38.20000076293945
Iteration 200, Epoch 1, Loss: 2.0732624530792236, Accuracy: 32.299442291259766, Val Loss: 1.8049792051315308, Val Accuracy: 40.29999923706055
Iteration 300, Epoch 1, Loss: 1.998197078704834, Accuracy: 34.15178680419922, Val Loss: 1.8227159976959229, Val Accuracy: 38.400001525878906
Iteration 400, Epoch 1, Loss: 1.9275578260421753, Accuracy: 36.050498962402344, Val Loss: 1.69352126121521, Val Accuracy: 43.79999923706055
Iteration 500, Epoch 1, Loss: 1.8840745687484741, Accuracy: 36.991641998291016, Val Loss: 1.6495952606201172, Val Accuracy: 43.900001525878906
Iteration 600, Epoch 1, Loss: 1.852650761604309, Accuracy: 37.91597366333008, Val Loss: 1.6607578992843628, Val Accuracy: 43.79999923706055
Iteration 700, Epoch 1, Loss

Поэкспериментируйте с архитектурой сверточной сети. Для вашего набора данных вам необходимо получить как минимум 70% accuracy на валидационной выборке за 10 эпох обучения. Опишите все эксперименты и сделайте выводы (без выполнения данного пункта работы приниматься не будут).

Эспериментируйте с архитектурой, гиперпараметрами, функцией потерь, регуляризацией, методом оптимизации.  

https://www.tensorflow.org/versions/r2.0/api_docs/python/tf/keras/layers/BatchNormalization#methods https://www.tensorflow.org/versions/r2.0/api_docs/python/tf/keras/layers/Dropout#methods

Для проведения эксперимента, параметризуем текущий класс. Так можно будет протестировать несколько различных конфигураций и систематизировать сравнения.

In [17]:
class CustomConvNet(tf.keras.Model):
    def __init__(self,
                 channels=(32, 64, 128),
                 fc_dim=256,
                 conv_dropout=0.25,
                 fc_dropout=0.5,
                 num_classes=10):
        super(CustomConvNet, self).__init__()
        ############################################################################
        # TODO: Construct a model that performs well on CIFAR-10                   #
        ############################################################################
        # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

        initializer = tf.initializers.VarianceScaling(scale=2.0)

        c1, c2, c3 = channels

        self.conv1 = tf.keras.layers.Conv2D(c1, 3, padding='same',
                                            kernel_initializer=initializer)
        self.bn1 = tf.keras.layers.BatchNormalization()

        self.conv2 = tf.keras.layers.Conv2D(c1, 3, padding='same',
                                            kernel_initializer=initializer)
        self.bn2 = tf.keras.layers.BatchNormalization()
        self.pool1 = tf.keras.layers.MaxPooling2D(pool_size=2, strides=2)
        self.drop1 = tf.keras.layers.Dropout(conv_dropout)

        self.conv3 = tf.keras.layers.Conv2D(c2, 3, padding='same',
                                            kernel_initializer=initializer)
        self.bn3 = tf.keras.layers.BatchNormalization()

        self.conv4 = tf.keras.layers.Conv2D(c2, 3, padding='same',
                                            kernel_initializer=initializer)
        self.bn4 = tf.keras.layers.BatchNormalization()
        self.pool2 = tf.keras.layers.MaxPooling2D(pool_size=2, strides=2)
        self.drop2 = tf.keras.layers.Dropout(conv_dropout)

        self.conv5 = tf.keras.layers.Conv2D(c3, 3, padding='same',
                                            kernel_initializer=initializer)
        self.bn5 = tf.keras.layers.BatchNormalization()
        self.pool3 = tf.keras.layers.MaxPooling2D(pool_size=2, strides=2)
        self.drop3 = tf.keras.layers.Dropout(conv_dropout)

        self.flatten = tf.keras.layers.Flatten()
        self.fc1 = tf.keras.layers.Dense(fc_dim, activation='relu',
                                         kernel_initializer=initializer)
        self.drop_fc = tf.keras.layers.Dropout(fc_dropout)
        self.fc2 = tf.keras.layers.Dense(num_classes, activation='softmax',
                                         kernel_initializer=initializer)

        # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
        ############################################################################
        #                            END OF YOUR CODE                              #
        ############################################################################

    def call(self, input_tensor, training=False):
        ############################################################################
        # TODO: Construct a model that performs well on CIFAR-10                   #
        ############################################################################
        # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

        x = self.conv1(input_tensor)
        x = self.bn1(x, training=training)
        x = tf.nn.relu(x)

        x = self.conv2(x)
        x = self.bn2(x, training=training)
        x = tf.nn.relu(x)
        x = self.pool1(x)
        x = self.drop1(x, training=training)

        x = self.conv3(x)
        x = self.bn3(x, training=training)
        x = tf.nn.relu(x)

        x = self.conv4(x)
        x = self.bn4(x, training=training)
        x = tf.nn.relu(x)
        x = self.pool2(x)
        x = self.drop2(x, training=training)

        x = self.conv5(x)
        x = self.bn5(x, training=training)
        x = tf.nn.relu(x)
        x = self.pool3(x)
        x = self.drop3(x, training=training)

        x = self.flatten(x)
        x = self.fc1(x)
        x = self.drop_fc(x, training=training)
        x = self.fc2(x)

        # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
        ############################################################################
        #                            END OF YOUR CODE                              #
        ############################################################################
        return x

Теперь определим конфигурации.

In [21]:
configs = [
    {
        "name": "config1",
        "channels": (32, 64, 128),
        "fc_dim": 256,
        "conv_dropout": 0.25,
        "fc_dropout": 0.5,
        "optimizer": "adam",
        "learning_rate": 1e-3
    },
    {
        "name": "config2",
        "channels": (32, 64, 128),
        "fc_dim": 256,
        "conv_dropout": 0.25,
        "fc_dropout": 0.5,
        "optimizer": "adam",
        "learning_rate": 5e-4
    },
    {
        "name": "config3",
        "channels": (32, 64, 128),
        "fc_dim": 256,
        "conv_dropout": 0.25,
        "fc_dropout": 0.5,
        "optimizer": "sgd",
        "learning_rate": 1e-2
    },
        {
        "name": "config4",
        "channels": (32, 64, 128),
        "fc_dim": 256,
        "conv_dropout": 0.3,
        "fc_dropout": 0.6,
        "optimizer": "adam",
        "learning_rate": 1e-3
    },
    {
        "name": "config5",
        "channels": (64, 128, 256),
        "fc_dim": 512,
        "conv_dropout": 0.25,
        "fc_dropout": 0.5,
        "optimizer": "adam",
        "learning_rate": 1e-3
    }
]

In [22]:
import time

results = []
print_every = 700

for configs in configs:
    print("Model config:", configs["name"])

    def model_init_fn(configs=configs):
        return CustomConvNet(
            channels=configs["channels"],
            fc_dim=configs["fc_dim"],
            conv_dropout=configs["conv_dropout"],
            fc_dropout=configs["fc_dropout"]
        )

    def optimizer_init_fn(configs=configs):
        if configs["optimizer"] == "adam":
            return tf.keras.optimizers.Adam(learning_rate=configs["learning_rate"])
        elif configs["optimizer"] == "sgd":
            return tf.keras.optimizers.SGD(
                learning_rate=configs["learning_rate"],
                momentum=0.9,
                nesterov=True
            )

    # Дополнительно замеряем время, т.к. некоторые конфиги могут обучаться долго
    start_time = time.time()
    best_val_acc = train_part34(
        model_init_fn,
        optimizer_init_fn,
        num_epochs=10,
        is_training=True
    )
    end_time = time.time()
    total_time = end_time - start_time

    results.append({
        "name": configs["name"],
        "best_val_acc": best_val_acc,
        "channels": configs["channels"],
        "fc_dim": configs["fc_dim"],
        "conv_dropout": configs["conv_dropout"],
        "fc_dropout": configs["fc_dropout"],
        "optimizer": configs["optimizer"],
        "learning_rate": configs["learning_rate"],
        "train_time_sec": round(total_time, 2),
        "train_time_min": round(total_time / 60, 2)
    })

print("Results:")
for r in results:
    print(r)

Model config: config1
Iteration 0, Epoch 1, Loss: 4.260814666748047, Accuracy: 14.0625, Val Loss: 5.038318157196045, Val Accuracy: 8.0
Iteration 700, Epoch 1, Loss: 1.8049647808074951, Accuracy: 32.417972564697266, Val Loss: 1.4570198059082031, Val Accuracy: 49.900001525878906
Iteration 1400, Epoch 2, Loss: 1.4441149234771729, Accuracy: 46.158958435058594, Val Loss: 1.1459428071975708, Val Accuracy: 60.000003814697266
Iteration 2100, Epoch 3, Loss: 1.255937099456787, Accuracy: 54.2948112487793, Val Loss: 1.1417806148529053, Val Accuracy: 60.20000076293945
Iteration 2800, Epoch 4, Loss: 1.1508140563964844, Accuracy: 58.48347473144531, Val Loss: 0.9290087819099426, Val Accuracy: 67.79999542236328
Iteration 3500, Epoch 5, Loss: 1.0660383701324463, Accuracy: 61.910037994384766, Val Loss: 0.8334011435508728, Val Accuracy: 70.4000015258789
Iteration 4200, Epoch 6, Loss: 1.0084280967712402, Accuracy: 64.17621612548828, Val Loss: 0.768785297870636, Val Accuracy: 72.79999542236328
Iteration 490

Опишите все эксперименты, результаты. Сделайте выводы.

### Описание эксперимента:
Сначала была протестирована базовая модель (простая трехслойная CNN). Однако её мощности недостаточно для высокого качества классификации. <br/>
В качестве эксперимента варьировались следующие параметры: количество каналов, добавлен Dropout, BatchNormalization, дополнительные сверточные слои и MaxPooling2D. <br/>
Для выявления лучшей модели были протестированы несколько вариантов наборов гиперпараметров: <br/>

| Эксперимент | Каналы Conv слоёв | FC слой | Dropout Conv | Dropout FC | Оптимизатор | Learning Rate |
|------------|-------------------|--------|--------------|------------|------------|---------------|
| 1 | (32, 64, 128) | 256 | 0.25 | 0.50 | Adam | 1e-3 |
| 2 | (32, 64, 128) | 256 | 0.25 | 0.50 | Adam | 5e-4 |
| 3 | (32, 64, 128) | 256 | 0.25 | 0.50 | SGD + Nesterov | 1e-2 |
| 4 | (32, 64, 128) | 256 | 0.30 | 0.60 | Adam | 1e-3 |
| 5 | (64, 128, 256) | 512 | 0.25 | 0.50 | Adam | 1e-3 |

<br/> Лучшей моделью оказалась конфигурация №5 с оптимизатором Adam, каналами (64, 128, 256), Learning Rate = 1e-3. Точность на валидационной выборке - 78.4%. <br/>

### Выводы:
1. После добавления BatchNormalization обучение стало значительно стабильнее. Уменьшились скачки loss, а также ускорилась сходимость и повысилась итоговая accuracy.
2. Dropout уменьшил переобучение. Несмотря на рост train accuracy, модель хорошо обобщала данные на validation set.
3. Pooling уменьшил размерность признаков и позволил извлекать более устойчивые пространственные признаки.
4. Оптимизатор Adam показал лучшую и более быструю сходимость по сравнению с SGD.
